# Minimal Text Classification: DistilBERT (Hugging Face Transformers)

**What this notebook does**
- Loads a CSV with columns `text` and `label`
- Keeps text raw (only trimming/empty-row filtering)
- Creates stratified train/val/test splits
- Tokenizes with `AutoTokenizer`
- Trains `AutoModelForSequenceClassification` (DistilBERT)
- Evaluates on test with accuracy + macro‑F1
- Saves model + tokenizer + label map

**Offline note**
- If your environment is offline, set `MODEL` to a **local path** containing a compatible DistilBERT checkpoint.
  E.g., `MODEL = '/path/to/distilbert-base-uncased'`.

**Input assumptions**
- Your CSV file path goes into `CSV_PATH` below.
- Two columns: `text` and `label`.

In [ ]:
#!pip install -q transformers datasets evaluate pandas scikit-learn
# If using conda, you can: conda install -c conda-forge transformers datasets evaluate pandas scikit-learn
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
import numpy as np
import evaluate

CSV_PATH = 'your_dataset.csv'  # <-- change me
RANDOM_STATE = 42
MODEL = 'distilbert-base-uncased'  # set to local path if offline


In [ ]:
# Load + minimal hygiene (no aggressive cleaning for BERT)
df = pd.read_csv(CSV_PATH)
assert {'text','label'}.issubset(df.columns), 'CSV must have columns: text,label'
df['text'] = df['text'].astype(str).str.strip()
df = df[df['text'].str.len() > 2].dropna(subset=['text','label'])
df = df.drop_duplicates(subset=['text','label'])
print('After hygiene:', df.shape)

# Label map
le = LabelEncoder()
df['label_id'] = le.fit_transform(df['label'])
print('Classes:', list(le.classes_))

# Stratified splits
train_df, temp_df = train_test_split(df, test_size=0.2, stratify=df['label_id'], random_state=RANDOM_STATE)
val_df, test_df   = train_test_split(temp_df, test_size=0.5, stratify=temp_df['label_id'], random_state=RANDOM_STATE)

ds = DatasetDict({
    'train': Dataset.from_pandas(train_df[['text','label_id']], preserve_index=False),
    'validation': Dataset.from_pandas(val_df[['text','label_id']], preserve_index=False),
    'test': Dataset.from_pandas(test_df[['text','label_id']], preserve_index=False),
})
ds

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL)

def tok(batch):
    return tokenizer(batch['text'], truncation=True, padding='max_length', max_length=256)

ds_tok = ds.map(tok, batched=True, remove_columns=['text'])
ds_tok = ds_tok.rename_column('label_id','labels')
ds_tok.set_format(type='torch', columns=['input_ids','attention_mask','labels'])
num_labels = len(le.classes_)
num_labels

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=num_labels)

acc = evaluate.load('accuracy')
f1m = evaluate.load('f1')

def compute_metrics(p):
    preds = np.argmax(p.predictions, axis=1)
    return {
        'accuracy': acc.compute(references=p.label_ids, predictions=preds)['accuracy'],
        'f1_macro': f1m.compute(references=p.label_ids, predictions=preds, average='macro')['f1']
    }

args = TrainingArguments(
    output_dir='distilbert_clf',
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    learning_rate=2e-5,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1_macro',
    report_to=[],
    seed=RANDOM_STATE,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=ds_tok['train'],
    eval_dataset=ds_tok['validation'],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)
trainer.train()


In [ ]:
# Final evaluation on held-out test set
test_metrics = trainer.evaluate(ds_tok['test'])
test_metrics

In [ ]:
# Save model + tokenizer + label map
trainer.model.save_pretrained('distilbert_clf/best')
tokenizer.save_pretrained('distilbert_clf/best')
pd.Series(le.classes_).to_csv('label_map.csv', index_label='id', header=['label'])
print('Saved to distilbert_clf/best and label_map.csv')
